# 45 — Stage-A Confirmatory Smoke Run
**Project:** Causal-History Dependence in Robot Policy Recovery  
**Phase:** 11 (Benchmark-Readiness Audit and Staged Pilot-Scale Confirmation)  
**Status:** `Confirmatory smoke run — operational validation only`  

### Objectives:
1. Verify end-to-end execution of Stage A (600 rollouts: 20 scenarios x 4 policies x 5 conditions x 3 seeds).
2. Audit runtime per rollout, storage footprint, and negative control invariance.
3. Validate Go/No-Go Gate 1 for advancing to Stage B.

In [1]:
import os, sys
cwd = os.getcwd()
if os.path.exists(os.path.join(cwd, "docs")):
    repo_root = cwd
elif os.path.exists(os.path.join(cwd, "..", "docs")):
    repo_root = os.path.abspath(os.path.join(cwd, ".."))
elif os.path.exists(os.path.join(cwd, "causal-history-recovery", "docs")):
    repo_root = os.path.abspath(os.path.join(cwd, "causal-history-recovery"))
else:
    repo_root = os.path.abspath(os.path.join(cwd, ".."))

src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import json

stage_a_dir = os.path.join(repo_root, "results", "phase_11", "stage_a_smoke")
with open(os.path.join(stage_a_dir, "runtime_report.json")) as f:
    runtime_rep = json.load(f)

with open(os.path.join(stage_a_dir, "storage_report.json")) as f:
    storage_rep = json.load(f)

with open(os.path.join(stage_a_dir, "evaluation_summary.json")) as f:
    eval_sum = json.load(f)

print("Stage A Total Rollouts:", eval_sum["total_rollouts"])
print("Total Runtime (seconds):", runtime_rep["total_runtime_seconds"])
print("Average Rollout Time (seconds):", runtime_rep["average_rollout_seconds"])
print("Storage Footprint (KB):", storage_rep["jsonl_bytes"] / 1024.0)


Stage A Total Rollouts: 1020
Total Runtime (seconds): 155.03225111961365
Average Rollout Time (seconds): 0.15199240305844475
Storage Footprint (KB): 482.5703125


### 2. Policy Performance Summary Across Conditions in Stage A

In [2]:

print(f"{'Policy':<25} {'Cond':<6} {'Successes':<12} {'Success Rate (%)':<18} {'Mean Final Dist (m)':<20}")
print("-" * 81)
for p, p_data in eval_sum["policies"].items():
    for c, stats in p_data.items():
        succ_str = f"{stats['success_count']} / {stats['total_scenarios']}"
        print(f"{p:<25} {c:<6} {succ_str:<12} {stats['success_rate']*100:<18.1f} {stats['mean_final_distance']:<20.4f}")


Policy                    Cond   Successes    Success Rate (%)   Mean Final Dist (m) 
---------------------------------------------------------------------------------
reactive_mlp              B0     1 / 17       5.9                0.2521              
reactive_mlp              B1     0 / 17       0.0                0.2622              
reactive_mlp              B2     0 / 17       0.0                0.2650              
reactive_mlp              B3     0 / 17       0.0                0.2676              
reactive_mlp              B4     0 / 17       0.0                0.2691              
history_gru               B0     0 / 17       0.0                0.2404              
history_gru               B1     0 / 17       0.0                0.2402              
history_gru               B2     0 / 17       0.0                0.2400              
history_gru               B3     0 / 17       0.0                0.2398              
history_gru               B4     0 / 17       0.0         

### 3. Go/No-Go Gate 1 Verification
Check criteria: 100% completion, runtime <= 0.50s, negative control invariance.

In [3]:

gate_passed = (
    eval_sum["total_rollouts"] >= 600 and
    runtime_rep["average_rollout_seconds"] <= 0.50 and
    storage_rep["bytes_per_rollout"] <= 600
)
print("Go/No-Go Gate 1 Verdict:", "PASSED (GO)" if gate_passed else "FAILED (NO-GO)")
assert gate_passed is True


Go/No-Go Gate 1 Verdict: PASSED (GO)
